In [1]:
from datasets import load_dataset

ds = load_dataset("legacy-datasets/banking77")
print(ds)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 10003
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 3080
    })
})


In [2]:
label_names = ds["train"].features["label"].names

print(f"Total intents: {len(label_names)}")
print(f"First 5: {label_names[:5]}")
print(f"Last 5:  {label_names[-5:]}")

Total intents: 77
First 5: ['activate_my_card', 'age_limit', 'apple_pay_or_google_pay', 'atm_support', 'automatic_top_up']
Last 5:  ['virtual_card_not_working', 'visa_or_mastercard', 'why_verify_identity', 'wrong_amount_of_cash_received', 'wrong_exchange_rate_for_cash_withdrawal']


In [3]:
import re

def decode_label(example):
    return {"intent": label_names[example["label"]]}

def clean_intent(name):
    name = re.sub(r'[?]', '', name)
    name = name.lower()
    return name

label_names = [clean_intent(n) for n in ds["train"].features["label"].names]

train_ds = ds["train"].map(decode_label)
test_ds  = ds["test"].map(decode_label)

train_ds = train_ds.remove_columns(["label"])
test_ds  = test_ds.remove_columns(["label"])

print(train_ds)
print(train_ds[0])

Map:   0%|          | 0/10003 [00:00<?, ? examples/s]

Map:   0%|          | 0/3080 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'intent'],
    num_rows: 10003
})
{'text': 'I am still waiting on my card?', 'intent': 'card_arrival'}


In [4]:
from collections import Counter

train_counts = Counter(train_ds["intent"])
test_counts  = Counter(test_ds["intent"])

print(sorted(train_counts.items(), key=lambda x: x[1]))

train_vals = list(train_counts.values())
test_vals  = list(test_counts.values())

print("TRAIN — min: {}, max: {}, mean: {:.1f}".format(min(train_vals), max(train_vals), sum(train_vals)/len(train_vals)))
print("TEST  — min: {}, max: {}, mean: {:.1f}".format(min(test_vals),  max(test_vals),  sum(test_vals)/len(test_vals)))

if min(train_vals) != max(train_vals):
    print(f"⚠️  Train imbalance — min: {min(train_vals)}, max: {max(train_vals)}")
else:
    print("✓ Train split perfectly balanced.")

[('contactless_not_working', 35), ('virtual_card_not_working', 41), ('card_acceptance', 59), ('card_swallowed', 61), ('lost_or_stolen_card', 82), ('compromised_card', 86), ('atm_support', 87), ('receiving_money', 95), ('top_up_limits', 97), ('get_disposable_virtual_card', 97), ('getting_virtual_card', 98), ('unable_to_verify_identity', 102), ('topping_up_by_card', 103), ('verify_my_identity', 104), ('passcode_forgotten', 105), ('get_physical_card', 106), ('terminate_account', 108), ('age_limit', 110), ('top_up_by_bank_transfer_charge', 111), ('exchange_rate', 112), ('card_delivery_estimate', 112), ('card_not_working', 112), ('verify_source_of_funds', 113), ('transfer_into_account', 113), ('top_up_by_cash_or_cheque', 114), ('top_up_by_card_charge', 114), ('pin_blocked', 115), ('exchange_via_app', 118), ('order_physical_card', 120), ('edit_personal_details', 121), ('why_verify_identity', 121), ('disposable_card_limits', 121), ('lost_or_stolen_phone', 121), ('exchange_charge', 121), ('cha

In [5]:
import random
from collections import defaultdict

random.seed(42)

intent_to_texts = defaultdict(list)
for example in train_ds:
    intent_to_texts[example["intent"]].append(example["text"])

print("Intents with only 1 sample (can't form a pair):")
singletons = [k for k, v in intent_to_texts.items() if len(v) < 2]
print(singletons if singletons else "  None: all intents have ≥2 samples ✓")

Intents with only 1 sample (can't form a pair):
  None: all intents have ≥2 samples ✓


In [6]:
pairs = []

for example in train_ds:
    anchor = example["text"]
    intent = example["intent"]
    candidates = [t for t in intent_to_texts[intent] if t != anchor]

    if not candidates:
        continue

    positive = random.choice(candidates)
    pairs.append({"anchor": anchor, "positive": positive, "intent": intent})

print(f"Total pairs built: {len(pairs)}")
print(f"Example pair:")
print(f"  anchor  : {pairs[0]['anchor']}")
print(f"  positive: {pairs[0]['positive']}")
print(f"  intent  : {pairs[0]['intent']}")

Total pairs built: 10003
Example pair:
  anchor  : I am still waiting on my card?
  positive: What is the expected delivery date for my card?
  intent  : card_arrival


In [7]:
from datasets import Dataset

pairs_ds = Dataset.from_list(pairs)
print(pairs_ds)
print(pairs_ds[0])

Dataset({
    features: ['anchor', 'positive', 'intent'],
    num_rows: 10003
})
{'anchor': 'I am still waiting on my card?', 'positive': 'What is the expected delivery date for my card?', 'intent': 'card_arrival'}


In [8]:
pairs_ds.save_to_disk("train_pairs")
test_ds.save_to_disk("test")

print("Saved:")
print("  data/train_pairs  — anchor/positive pairs for MNR training")
print("  data/test         — official test split, untouched, for evaluation only")

Saving the dataset (0/1 shards):   0%|          | 0/10003 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3080 [00:00<?, ? examples/s]

Saved:
  data/train_pairs  — anchor/positive pairs for MNR training
  data/test         — official test split, untouched, for evaluation only


In [9]:
from datasets import load_from_disk

train_pairs = load_from_disk("train_pairs")
test = load_from_disk("test")

print("=== Dataset Summary ===")
print(f"Train pairs : {len(train_pairs):>6}  (anchor + positive per sample)")
print(f"Test samples: {len(test):>6}  (text + intent, evaluation only)")
print(f"Intents     : {len(label_names):>6}")
print(f"Train per intent (pairs): {len(train_pairs) // len(label_names)}")
print(f"Test  per intent        : {len(test) // len(label_names)}")

=== Dataset Summary ===
Train pairs :  10003  (anchor + positive per sample)
Test samples:   3080  (text + intent, evaluation only)
Intents     :     77
Train per intent (pairs): 129
Test  per intent        : 40
